# Law RAG baseline

BM25 (лемматизация) + опционально полный пайплайн из `src/retriever.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from preprocess import tokenize_lemmas

docs = pd.read_csv(ROOT / 'data' / 'documents.csv')
train = pd.read_csv(ROOT / 'data' / 'train.csv')
test = pd.read_csv(ROOT / 'data' / 'test.csv')

corpus = [tokenize_lemmas(t) for t in docs.text]
bm25 = BM25Okapi(corpus)
doc_ids = docs.doc_id.tolist()

def top5(question: str):
    scores = bm25.get_scores(tokenize_lemmas(question))
    return [doc_ids[i] for i in np.argsort(scores)[::-1][:5]]

hits = sum(g in top5(q) for q, g in zip(train.question, train.gold_doc_id))
print('BM25 train Recall@5:', hits / len(train))

rows = [{'qid': qid, 'doc_id': d} for qid, q in zip(test.qid, test.question) for d in top5(q)]
sub = pd.DataFrame(rows)
sub.to_csv(ROOT / 'outputs' / 'submission_bm25.csv', index=False)
sub.head(10)

## Полный пайплайн

```bash
python src/retriever.py --skip-cv
```

Результат: `submission.csv` (hybrid BM25 + e5 + LightGBM LambdaRank).